In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import numpy as np


In [ ]:
# Transformações
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Carregar dataset completo
data_dir = "/home/lobo/github/RepLearningLosses/PlatonicSolidsImages"
dataset = datasets.ImageFolder(root=data_dir, transform=transform)

# Organizar índices por classe
class_indices = {cls: [] for cls in range(len(dataset.classes))}
for idx, (_, label) in enumerate(dataset.samples):
    class_indices[label].append(idx)

# Selecionar 30% dos dados para treino
train_indices = []
remaining_indices = []

for cls, indices in class_indices.items():
    np.random.shuffle(indices)  # Embaralhar para evitar viés
    split_idx = int(0.3 * len(indices))  # 30% dos dados dessa classe
    train_indices.extend(indices[:split_idx])
    remaining_indices.extend(indices[split_idx:])  # O restante vai para validação/teste

# Criar conjuntos de treino, validação e teste
train_dataset = Subset(dataset, train_indices)

# Dividir o restante em validação (50%) e teste (50%)
val_size = len(remaining_indices) // 2
val_dataset = Subset(dataset, remaining_indices[:val_size])
test_dataset = Subset(dataset, remaining_indices[val_size:])

# Criar DataLoaders
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

print(f"Treino: {len(train_dataset)} imagens")
print(f"Validação: {len(val_dataset)} imagens")
print(f"Teste: {len(test_dataset)} imagens")


In [ ]:
len(dataset)

In [ ]:
from collections import Counter

# Obter rótulos originais
all_labels = [label for _, label in dataset.samples]

# Contar classes no conjunto de treino
train_labels = [all_labels[idx] for idx in train_indices]
train_class_counts = Counter(train_labels)

# Contar classes no conjunto de validação
val_labels = [all_labels[idx] for idx in remaining_indices[:val_size]]
val_class_counts = Counter(val_labels)

# Contar classes no conjunto de teste
test_labels = [all_labels[idx] for idx in remaining_indices[val_size:]]
test_class_counts = Counter(test_labels)

# Exibir os resultados
print("\nDistribuição das classes:")
for cls, class_name in enumerate(dataset.classes):
    print(f"Classe '{class_name}':")
    print(f"  ➤ Treino: {train_class_counts.get(cls, 0)} imagens")
    print(f"  ➤ Validação: {val_class_counts.get(cls, 0)} imagens")
    print(f"  ➤ Teste: {test_class_counts.get(cls, 0)} imagens")


In [1]:
from torch.utils.data import DataLoader
from torchvision import transforms
from util import CustomDatasetFromCSV
from util import AverageMeter, DoubleTransform, SubsetWithTargets, TwoCropTransform
import numpy as np

root_path = ""
train_files = 'Datasets/KFolds/SKF_TRAIN_Fold_1.csv'
val_files = 'Datasets/KFolds/SKF_VAL_Fold_1.csv'

In [2]:
image_size = (224, 224)
test_transform = DoubleTransform(
                    transforms.Compose([
                        transforms.Resize(image_size),
                        transforms.ToTensor(),
                        transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2675, 0.2565, 0.2761]),
                    ]),
                    transforms.Compose([
                        transforms.RandomResizedCrop(size=image_size, scale=(0.2, 1.)),
                        transforms.RandomHorizontalFlip(),
                        transforms.ToTensor(),
                        transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2675, 0.2565, 0.2761])
                ]))

# both images heavily augmented
train_transform = TwoCropTransform(transforms.Compose([
                    transforms.RandomResizedCrop(size=image_size, scale=(0.2, 1.)),
                    transforms.RandomHorizontalFlip(),
                    transforms.RandomApply([
                        transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)
                    ], p=0.8),
                    transforms.RandomGrayscale(p=0.2),
                    transforms.ToTensor(),
                    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2675, 0.2565, 0.2761])
                ]))

In [ ]:
train = CustomDatasetFromCSV(root_path, tf_image=None, csv_name=train_files, task=None, as_rgb=True)
test = CustomDatasetFromCSV(root_path, tf_image=None, csv_name=val_files, task=None, as_rgb=True)

In [4]:
train_loader = DataLoader(train, batch_size=32, shuffle=True, num_workers=4)
test_loader = DataLoader(test, batch_size=32, shuffle=False, num_workers=4)

In [5]:
train_loader.dataset[2]

(<PIL.Image.Image image mode=RGB size=224x224>, 0)

In [6]:
len(train_loader.dataset)

38313

In [7]:
train_transfomations = CustomDatasetFromCSV(root_path, tf_image=train_transform, csv_name=train_files, task=None, as_rgb=True)
test_transformations = CustomDatasetFromCSV(root_path, tf_image=test_transform, csv_name=val_files, task=None, as_rgb=True)

In [10]:
train_loader_trans = DataLoader(train_transfomations, batch_size=32, shuffle=True, num_workers=4)
test_loader_trans = DataLoader(test_transformations, batch_size=32, shuffle=False, num_workers=4)

In [11]:
len(train_loader_trans.dataset)

38313

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

def show_contrastive_images(dataset_loader):
    """Function to show positive pairs of images from contrastive learning dataset.

    Assumes the dataloader returns (view1, view2, labels).
    """
    batch = next(iter(dataset_loader))  # Obtém um batch do DataLoader
    view1, view2, labels = batch  # Cada amostra contém duas views da mesma imagem

    # Número de imagens a serem exibidas (limitado para evitar visualização muito grande)
    num_images = min(5, view1.shape[0])

    fig, axes = plt.subplots(num_images, 2, figsize=(6, 3 * num_images))

    for i in range(num_images):
        img1 = view1[i].cpu().permute(1, 2, 0).numpy()  # Converte para [H, W, C]
        img2 = view2[i].cpu().permute(1, 2, 0).numpy()  # Converte para [H, W, C]

        # Normaliza para [0,1] se necessário
        img1 = (img1 - img1.min()) / (img1.max() - img1.min())
        img2 = (img2 - img2.min()) / (img2.max() - img2.min())

        axes[i, 0].imshow(img1)
        axes[i, 0].axis("off")
        axes[i, 0].set_title(f"View 1 (Label: {labels[i].item()})")

        axes[i, 1].imshow(img2)
        axes[i, 1].axis("off")
        axes[i, 1].set_title(f"View 2 (Label: {labels[i].item()})")

    plt.tight_layout()
    plt.show()

# Exemplo de uso:
show_contrastive_images(train_loader)


In [ ]:
import matplotlib.pyplot as plt
from torchvision.utils import make_grid
import os

def show_images(dataset_loader, db_name, path_to_save):
    """function that show images from dataloader

    Args:
        dataset_loader (torch.utils.data.Dataloader): images dataloader
        db_name (str): database name
        path_to_save (str): path to save images

    """
    os.makedirs(path_to_save, exist_ok=True)
    batch = next(iter(dataset_loader[0]))
    images, labels = batch
        
    plt.figure(figsize=(11, 11))
    plt.axis("off")
    plt.title("Training Images")
    plt.imshow(np.transpose(make_grid(images[:32], padding=2, normalize=True), (1, 2, 0)))
    plt.savefig(os.path.join(path_to_save, "preview_train_{}.png".format(db_name)))

In [ ]:
show_images(train_loader[0], "ModelNet10", "Datasets/figures")

In [12]:
import pandas as pd

dataset = pd.read_csv("Datasets/train_folds.csv")
dataset.head()

,image_path,label,fold
0,ModelNet10/train/bathtub/bathtub_0020_4.jpg,bathtub,0
1,ModelNet10/train/bathtub/bathtub_0026_8.jpg,bathtub,3
2,ModelNet10/train/bathtub/bathtub_0055_1.jpg,bathtub,2
3,ModelNet10/train/bathtub/bathtub_0060_11.jpg,bathtub,4
4,ModelNet10/train/bathtub/bathtub_0044_0.jpg,bathtub,2


In [ ]:
from util import CustomDatasetFromCSV
from torch.utils.data import DataLoader
from torchvision import transforms

fold = 3
data_fold = dataset[dataset["fold"] != fold]
# data_fold.index.to_list()
# data_fold.count()
# data_fold.head()
# data_fold.index.to_list()
# type(data_fold)
# print(data_fold.index)
# print(data_fold.index.to_list())
data_fold
38314/32

9578.5